<a href="https://colab.research.google.com/github/selim679/databricks-streaming-pipeline/blob/main/02_transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


df_bronze = spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/bronze")


df_silver = df_bronze \
    .filter(col("user_id").isNotNull()) \
    .filter(col("event_type").isin("click", "view", "purchase")) \
    .withColumn("amount", col("amount").cast("double")) \
    .withColumn("event_date", to_date(col("timestamp"))) \
    .dropDuplicates(["user_id", "timestamp"])


df_silver = df_silver.filter(col("amount") >= 0)

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("event_date") \
    .save("/Volumes/workspace/default/youtubeseries/delta/silver")

In [ ]:
df = spark.read.format("delta").load("/Volumes/workspace/default/youtubeseries/delta/silver")
display(df)

In [ ]:
print(f"Rows after cleaning: {df_silver.count()}")

In [ ]:
df_silver.select([count(when(col(c).isNull(), c)).alias(c) for c in df_silver.columns]).show()

In [ ]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .save("/Volumes/workspace/default/youtubeseries/delta/silver")